In [1]:
import json
import requests
import pandas as pd

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
# 1) Cargar API key desde el JSON
with open("../config/api_config.json", "r", encoding="utf-8") as f:
    api_key = json.load(f)["aemet_api_key"]

In [5]:
# 2) Endpoint que quieres consultar
idema = "3195" 
url_api = f"https://opendata.aemet.es/opendata/api/observacion/convencional/datos/estacion/{idema}"


# url_estaciones = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/estaciones/{estaciones}"
# url_estaciones_todas = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones"


In [6]:
# 3) Primera llamada: obtiene la URL "datos"
resp = requests.get(url_api, params={"api_key": api_key}, timeout=30)
resp.raise_for_status()
meta = resp.json()

# AEMET suele devolver: estado, descripcion, datos, metadatos...
if "datos" not in meta:
    raise RuntimeError(f"No se encontró 'datos' en la respuesta: {meta}")

datos_url = meta["datos"]

In [7]:
datos_url

'https://opendata.aemet.es/opendata/sh/894c835f'

In [ ]:

# 4) Segunda llamada: descarga los datos reales (robusta)

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)



headers = {
    "Accept": "application/json",
    "User-Agent": "python-requests aemet-client"
}

resp_datos = session.get(datos_url, headers=headers, timeout=30)
resp_datos.raise_for_status()

try:
    prediccion = resp_datos.json()
except ValueError:
    prediccion = resp_datos.text

In [11]:
prediccion

[{'idema': '3195',
  'lon': -3.678095,
  'fint': '2026-01-21T02:00:00+0000',
  'prec': 0.0,
  'alt': 667.0,
  'vmax': 3.0,
  'vv': 0.5,
  'dv': 231.0,
  'lat': 40.411806,
  'dmax': 222.0,
  'ubi': 'MADRID RETIRO',
  'pres': 936.0,
  'hr': 95.0,
  'pres_nmar': 1015.9,
  'tamin': 2.0,
  'ta': 2.1,
  'tamax': 2.3,
  'tpr': 1.5,
  'rviento': 38.0},
 {'idema': '3195',
  'lon': -3.678095,
  'fint': '2026-01-21T03:00:00+0000',
  'prec': 0.0,
  'alt': 667.0,
  'vmax': 2.3,
  'vv': 1.1,
  'dv': 127.0,
  'lat': 40.411806,
  'dmax': 121.0,
  'ubi': 'MADRID RETIRO',
  'pres': 935.6,
  'hr': 95.0,
  'pres_nmar': 1015.5,
  'tamin': 1.7,
  'ta': 1.9,
  'tamax': 2.1,
  'tpr': 1.3,
  'rviento': 24.0},
 {'idema': '3195',
  'lon': -3.678095,
  'fint': '2026-01-21T04:00:00+0000',
  'prec': 0.0,
  'alt': 667.0,
  'vmax': 2.4,
  'vv': 1.1,
  'dv': 165.0,
  'lat': 40.411806,
  'dmax': 140.0,
  'ubi': 'MADRID RETIRO',
  'pres': 935.1,
  'hr': 95.0,
  'pres_nmar': 1015.1,
  'tamin': 1.4,
  'ta': 1.4,
  'tamax'

In [ ]:


root = prediccion[0]

municipio_id = root.get("id")
municipio = root.get("nombre")
provincia = root.get("provincia")
elaborado = root.get("elaborado")

dias = root["prediccion"]["dia"]

# -------------------------
# Helpers
# -------------------------
def to_map(lst, key="periodo", val="value"):
    """Convierte [{'periodo':'08','value':'5'}, ...] en {'08':'5', ...}"""
    out = {}
    for x in lst or []:
        k = x.get(key)
        if k is None:
            continue
        out[k] = x.get(val)
    return out

def to_map_desc(lst):
    """estadoCielo -> {'08': {'value':..., 'descripcion':...}, ...}"""
    out = {}
    for x in lst or []:
        p = x.get("periodo")
        if p is None:
            continue
        out[p] = {"value": x.get("value"), "descripcion": x.get("descripcion")}
    return out

def parse_viento_and_racha(lst):
    """
    vientoAndRachaMax viene intercalado:
      {'direccion': ['E'], 'velocidad': ['2'], 'periodo': '08'},
      {'value': '7', 'periodo': '08'},
    Devuelve 3 mapas: dir_map, vel_map, racha_map
    """
    dir_map, vel_map, racha_map = {}, {}, {}
    for x in lst or []:
        p = x.get("periodo")
        if p is None:
            continue
        if "direccion" in x or "velocidad" in x:
            # direccion y velocidad vienen como listas de strings
            d = x.get("direccion")
            v = x.get("velocidad")
            dir_map[p] = d[0] if isinstance(d, list) and d else d
            vel_map[p] = v[0] if isinstance(v, list) and v else v
        else:
            # racha máxima en 'value'
            racha_map[p] = x.get("value")
    return dir_map, vel_map, racha_map

# -------------------------
# 1) TABLA DÍA
# -------------------------
rows_dia = []
for d in dias:
    rows_dia.append({
        "municipio_id": municipio_id,
        "municipio": municipio,
        "provincia": provincia,
        "elaborado": elaborado,
        "fecha": pd.to_datetime(d["fecha"]).date(),
        "orto": d.get("orto"),
        "ocaso": d.get("ocaso"),
    })

df_dia = pd.DataFrame(rows_dia)

# -------------------------
# 2) TABLA HORA (00-23)
# -------------------------
rows_hora = []
for d in dias:
    fecha = pd.to_datetime(d["fecha"]).date()

    estado_map = to_map_desc(d.get("estadoCielo"))
    temp_map = to_map(d.get("temperatura"))
    st_map = to_map(d.get("sensTermica"))
    hum_map = to_map(d.get("humedadRelativa"))
    prec_map = to_map(d.get("precipitacion"))
    nieve_map = to_map(d.get("nieve"))

    viento_dir_map, viento_vel_map, racha_map = parse_viento_and_racha(d.get("vientoAndRachaMax"))

    # union de horas presentes (a veces no vienen todas)
    horas = sorted(
        set()
        | set(estado_map.keys())
        | set(temp_map.keys())
        | set(st_map.keys())
        | set(hum_map.keys())
        | set(prec_map.keys())
        | set(nieve_map.keys())
        | set(viento_dir_map.keys())
        | set(viento_vel_map.keys())
        | set(racha_map.keys())
    )

    for h in horas:
        ec = estado_map.get(h, {})
        rows_hora.append({
            "municipio_id": municipio_id,
            "municipio": municipio,
            "provincia": provincia,
            "fecha": fecha,
            "hora": int(h),  # "08" -> 8
            "estadoCielo_value": ec.get("value"),
            "estadoCielo_desc": ec.get("descripcion"),
            "temperatura": temp_map.get(h),
            "sensTermica": st_map.get(h),
            "humedadRelativa": hum_map.get(h),
            "precipitacion": prec_map.get(h),
            "nieve": nieve_map.get(h),
            "viento_dir": viento_dir_map.get(h),
            "viento_vel": viento_vel_map.get(h),
            "racha_max": racha_map.get(h),
        })

df_hora = pd.DataFrame(rows_hora).sort_values(["fecha", "hora"]).reset_index(drop=True)

# (opcional) convertir numéricos a float
for c in ["temperatura","sensTermica","humedadRelativa","precipitacion","nieve","viento_vel","racha_max"]:
    if c in df_hora.columns:
        df_hora[c] = pd.to_numeric(df_hora[c], errors="coerce")

# -------------------------
# 3) TABLA PROBABILIDADES POR TRAMOS (0713, 1319, 1901...)
# -------------------------
rows_tramos = []
tramo_keys = ["probPrecipitacion", "probTormenta", "probNieve"]

for d in dias:
    fecha = pd.to_datetime(d["fecha"]).date()
    for k in tramo_keys:
        for item in d.get(k, []) or []:
            rows_tramos.append({
                "municipio_id": municipio_id,
                "municipio": municipio,
                "provincia": provincia,
                "fecha": fecha,
                "tipo": k,
                "periodo_tramo": item.get("periodo"),
                "value": pd.to_numeric(item.get("value"), errors="coerce")
            })

df_tramos = pd.DataFrame(rows_tramos).sort_values(["fecha", "tipo", "periodo_tramo"]).reset_index(drop=True)


In [ ]:
df_dia

In [ ]:
df_hora.head()

In [ ]:
df_tramos